# Exploratory Data Analysis

**Explored dataset:** [Heart Disease](https://archive.ics.uci.edu/dataset/45/heart+disease)

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
heart_disease = fetch_ucirepo(id=45)

## Feature analysis

In [ ]:
X: pd.DataFrame = heart_disease.data.features 

X

In [ ]:
X.info()

Every single column is represented a number. Even the categorical ones :(.

Almost no missing values. The only columns with data missing are:
- `ca` (**4** missing) 
- `thal` (**2** missing)

I think that we can just remove these records.

### Categorical data

In [ ]:
# On a website it says that ca is a numerical columns but I think that It Can be analised as a categorical one.  
cat_labels = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

# Convert it to strings so we can use the built in describe method.
X_cat = X[cat_labels].astype(str)

X_cat

In [ ]:
X_cat.describe()

In [ ]:
X_cat[X_cat['ca'].isna()]

In [ ]:
X_cat[X_cat['thal'].isna()]

Over all there are only 6 records with missing values.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15,8))
axes = axes.flatten()

for i, col in enumerate(X_cat.columns):
    counts = X_cat[col].value_counts(dropna=False).sort_index()

    x_labels = [str(x) for x in counts.index]

    axes[i].bar(x_labels, counts.values, color='skyblue', edgecolor='black')
    axes[i].set_title(f"{col}")
    axes[i].set_ylabel("Count")
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


None of the distibutions aren't evened out. It's not even close

### Numerical data

In [ ]:
num_labels = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

X_num = X[num_labels]

X_num

In [ ]:
X_num.describe()

No missing values here

Small standard deviation relative to mean. Only exception is oldpeak where std is larger than its mean (data is spread).

Data not normalized

In [ ]:
X_num.hist(bins=20, figsize=(12,8))

**age** does not resemble the normal distibution. There are multiple peaks on this histogram. 

**trestbps**, **chol** and **thalach** are some normal deistribution-esque. **chol** and **trestbps** are shifted to the left and **thalach** is shifted to the right.

**oldpeak** looks like exponential distribution. Might be worth to **log** it out.

In [ ]:
sns.heatmap(X_num.corr(), annot=True, vmin=1.0, vmax=-1.0)

**age** has big correlation values with other fields. 

What stands out is also the correlation between **thalach** and **oldpeak**

## Target analysis

In [ ]:
y: pd.DataFrame = heart_disease.data.targets

y

In [ ]:
y.info()

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(6,6))

y_counts = y['num'].value_counts(dropna=False).sort_index()

y_labels = [str(a) for a in y_counts.index]

axes.bar(y_labels, y_counts.values, color='skyblue', edgecolor='black')
axes.set_title("target")
axes.set_ylabel("Count")
axes.tick_params(axis='y', rotation=45)

plt.tight_layout()
plt.show()

This is so not balanced. I migh have to duplicate the records.

In [ ]:
y_bin = (y>0).astype(int)


fig, axes = plt.subplots(1, 1, figsize=(6,6))

y_counts_bin = y_bin['num'].value_counts(dropna=False).sort_index()

y_labels_bin = [str(a) for a in y_counts_bin.index]

axes.bar(y_labels_bin, y_counts_bin.values, color='skyblue', edgecolor='black')
axes.set_title("target")
axes.set_ylabel("Count")
axes.tick_params(axis='y', rotation=45)

plt.tight_layout()
plt.show()

Much better

In [ ]:
X.corrwith(y.num)

Huge correlations here. Especially with the **oldpeak**, **cp** and **exang**.

There are also less important ones. **fbs** and **chol** to be exact.